# 1.Experiment Design and Background Introduction

## Dataset and Task background
This project studies the task of toxic comment classification using the **Jigsaw Toxic Comment dataset**, which is commonly used to benchmark the automatic detection of harmful language in user-generated comments. Each comment may be associated with multiple categories, such as toxicity, insults, threats, obscene content, and identity-based hate, making the task **multi-label text classification** problem.

This task belongs to the broader area of **content moderation**, rather than sentiment or emotion analysis. In practical systems, the objective is not to understand how a user feels, but to detect content that violates platform policies and may require intervention. As a result, errors are not equally costly: missing a genuinely harmful comment (false negative) is often more problematic than flagging a borderline case (false positive).

## Model Selection
We deliberately select **two models** from fundamentally different modelling paradigms.

1. The first is a **statistical**, feature-based model using **TF-IDF representations and Logistic Regression**. TF-IDF provides a more informative representation than simple count-based or binary bag-of-words models by weighting terms according to both their local frequency and global importance across the corpus. Logistic Regression is then used as a linear discriminative classifier that operates directly on this sparse, high-dimensional feature space. Compared to other statistical alternatives, this combination strikes a deliberate balance. It avoids the strong independence assumptions of Naive Bayes, retains probabilistic outputs and a convex optimisation objective unlike margin-based models such as linear SVMs, and remains well suited to sparse text representations where tree-based methods are less effective. At the same time, the linear structure of Logistic Regression allows direct inspection of feature weights, providing clear global interpretability. Together, TF-IDF and Logistic Regression form a strong and principled statistical baseline that captures lexical information as effectively as possible without introducing contextual modelling.

2. The second model is a **neural**, contextual model based on fine-tuning **DistilBERT** for multi-label classification. While several transformer variants such as BERT or RoBERTa could be applied to this task, DistilBERT is selected as a deliberate design choice rather than for maximal performance. The Jigsaw Toxic Comment dataset is moderate in size and consists of relatively short, direct
user comments, for which the full capacity of larger transformer models is unlikely to be necessary. DistilBERT retains most of BERT’s contextual modelling capability while using a substantially smaller architecture, reducing computational cost and limiting confounding effects from model size. This makes it a suitable representative of contextual transformer models when the goal is to isolate the impact of contextual information itself. Large language models are not considered, as the task is a supervised, discriminative classification
problem with abundant labelled data, where prompt-based or generative modelling would introduce additional complexity and reduce experimental control and interpretability without clear benefit.

## Objective

By comparing these two models, the project aims to highlight the trade-offs between statistical and neural approaches in a realistic content moderation setting, with a interest in sensitivity to rare labels and on the distinction between explicit lexical toxicity and more subtle, context-dependent harmful language.


# 2.Dataset and Evaluation Protocol

We use the official train/test split provided by the Jigsaw dataset.

Although cross-validation can provide robust estimates on small datasets,
it is not necessary here due to:
- the large training set (~160k samples),
- the availability of an official test split,
- and the desire to remain comparable with existing benchmarks.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report, f1_score

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# ----------------------------
# 1. Load data
# ----------------------------

# Load train and test separately
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/toxic_comment_dataset"

train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
test_df = pd.read_csv(f"{DATA_DIR}/test.csv")

label_cols = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

X_train = train_df["comment_text"].fillna("")
Y_train = train_df[label_cols]

X_test = test_df["comment_text"].fillna("")
Y_test = test_df[label_cols]

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)


Train size: (159571,)
Test size: (63978,)


Before training any model, we first inspect the provided **data split**. We briefly inspect the official data split to confirm its scale. The dataset contains approximately 160k training samples and 64k test samples, which is sufficient for reliable evaluation under the standard train/test protocol.

We therefore follow the official evaluation setting and use the provided train/test split throughout the project. While K-fold cross-validation can be useful for reducing estimation variance on smaller datasets or in the absence of a fixed test set, the scale of the data and the presence of a dedicated test set already satisfy these requirements in this case.


# 3.Statistical Baseline: TF-IDF + Logistic Regression

We first implement a classical statistical model based on TF-IDF features
and Logistic Regression in a one-vs-rest setting.

This model serves as:
- a strong and widely used baseline for text classification,
- a highly interpretable reference point,
- a contrast to neural contextual models.


## 3.1 Model & Features

*   TF-IDF unigram + bigram
*   One-vs-Rest LR

We use TF-IDF features with unigrams and bigrams to capture both lexical cues and short contextual patterns.

In [4]:
# ----------------------------
# 2. Feature extraction (TF-IDF)
# ----------------------------
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),     # unigrams + bigrams
    max_features=20000,
    min_df=2,
    max_df=0.9,
    stop_words="english"
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("TF-IDF feature matrix:", X_train_tfidf.shape)

TF-IDF feature matrix: (159571, 20000)


In [5]:
# ----------------------------
# 3. Model: Logistic Regression (One-vs-Rest)
# ----------------------------
lr = LogisticRegression(
    max_iter=1000,
    solver="liblinear"
)

model = OneVsRestClassifier(lr)
model.fit(X_train_tfidf, Y_train)

OneVsRestClassifier(estimator=LogisticRegression(max_iter=1000,
                                                 solver='liblinear'))

Logistic Regression is used as a linear classifier operating on this sparse feature space, with one binary classifier trained per label to handle the multi-label setting.

In [6]:
# ----------------------------
# 4. Prediction
# ----------------------------
Y_pred = model.predict(X_test_tfidf)

## 3.2 Performance Analysis

Since this is a multi-label classification problem, we evaluate model performance using classification metrics. In particular, we report micro- and macro-averaged F1 scores, which are commonly used for imbalanced multi-label classification.

In [7]:
# ----------------------------
# 5. Evaluation
# ----------------------------
print("Micro-F1:",
      f1_score(Y_test, Y_pred, average="micro"))

print("Macro-F1:",
      f1_score(Y_test, Y_pred, average="macro"))

print("\nPer-label performance:\n")
print(classification_report(
    Y_test,
    Y_pred,
    target_names=label_cols
))

Micro-F1: 0.6370780298837853
Macro-F1: 0.49422634087765355

Per-label performance:

               precision    recall  f1-score   support

        toxic       0.65      0.70      0.67      6090
 severe_toxic       0.41      0.31      0.36       367
      obscene       0.75      0.62      0.68      3691
       threat       0.45      0.21      0.28       211
       insult       0.73      0.51      0.60      3427
identity_hate       0.71      0.25      0.37       712

    micro avg       0.68      0.60      0.64     14498
    macro avg       0.62      0.43      0.49     14498
 weighted avg       0.69      0.60      0.63     14498
  samples avg       0.06      0.05      0.06     14498



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


The baseline model achieves reasonable overall performance (micro-F1 = 0.64), with clear differences across labels.

Interestingly Performance is relatively **strong** for frequent and lexically explicit categories such as **toxic**, *o*bscene**, and **insult**, where clear offensive cues are often present. In contrast, F1 scores are notably **lower** for rare labels such as **threat** and **identity_hate**.

We next look at the label distribution in the training data to understand whether the observed imbalance in model performance mirrors the underlying data distribution.


In [8]:
# List of toxicity labels
label_cols = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]
label_distribution = train_df[label_cols].mean().sort_values(ascending=False)
print(label_distribution)

toxic            0.095844
obscene          0.052948
insult           0.049364
severe_toxic     0.009996
identity_hate    0.008805
threat           0.002996
dtype: float64


The label distribution confirms that the uneven performance observed for the TF-IDF + Logistic Regression baseline closely reflects the underlying data distribution.

General toxicity-related labels such as *toxic*, *obscene*, and *insult* are relatively frequent, whereas *threat* and *identity_hate* are extremely rare, accounting for less than 1% of the training instances. This strong class imbalance is consistent with the dataset’s original purpose for content moderation, where most user comments are not threats and explicit identity-based hate is uncommon. As a result, standard statistical classifiers tend to achieve lower recall and F1 scores for these rare labels, which aligns with the performance patterns observed above.

## 3.3 Interpretability: Global Feature Weights

For the TF-IDF + Logistic Regression baseline, we analyse model interpretability by inspecting feature weights. As Logistic Regression is a linear model, its predictions are directly determined by a weighted sum of input features. Therefore, the **learned coefficients** provide a faithful and transparent explanation of which words or n-grams contribute most strongly to each label.

This form of global interpretability is particularly appropriate for
feature-based statistical models, and does not require additional post-hoc explanation methods.

In [ ]:
# ----------------------------
# 6. Interpretability
# ----------------------------
feature_names = np.array(vectorizer.get_feature_names_out())

for i, label in enumerate(label_cols):
    coef = model.estimators_[i].coef_[0]

    top_positive = feature_names[np.argsort(coef)[-10:]]

    print(f"\nTop features for label '{label}':")
    print(top_positive)



Top features for label 'toxic':
['bitch' 'suck' 'asshole' 'bullshit' 'ass' 'stupid' 'shit' 'idiot'
 'fucking' 'fuck']

Top features for label 'severe_toxic':
['die' 'fucker' 'suck' 'dick' 'bitch' 'fuckin' 'asshole' 'motherfucker'
 'fucking' 'fuck']

Top features for label 'obscene':
['dick' 'cunt' 'suck' 'bullshit' 'shit' 'asshole' 'ass' 'bitch' 'fucking'
 'fuck']

Top features for label 'threat':
['burn' 'fucking' 'destroy' 'shoot' 'going' 'ass' 'rape' 'death' 'die'
 'kill']

Top features for label 'insult':
['ass' 'moron' 'idiots' 'fuck' 'fucking' 'faggot' 'stupid' 'bitch'
 'asshole' 'idiot']

Top features for label 'identity_hate':
['jews' 'nazi' 'homo' 'jew' 'faggot' 'niggers' 'nigga' 'homosexual' 'gay'
 'nigger']


Inspecting the highest-weighted features provides a clear and faithful global interpretation of the TF-IDF + Logistic Regression model. Three main observations emerge.

1.**The model relies on explicit and human-interpretable lexical cues**

Across labels, **the highest-weighted features consist of explicit offensive terms** that closely align with human intuition and with the dataset’s annotation guidelines. For example, **general toxicity (toxic)** is associated with common insults such as bitch, asshole, stupid, and idiot, while **obscene** is dominated by sexual or physiological vulgarities such as dick, cunt, and shit. Similarly, **threat** is characterised by verbs denoting explicit violent actions (kill, shoot, burn, rape), and **identity_hate** is strongly linked to identity-related slurs (jew, gay, nigger).

These patterns indicate that the model’s decisions are grounded in clear, semantically meaningful lexical evidence rather than spurious correlations.

2.**The model captures coarse distinctions, but struggles with fine-grained ones.**

While the learned features reflect the intended semantics of each label, there is substantial overlap between labels. Highly **frequent offensive words** such as fuck and asshole appear among the top features for multiple categories (toxic, severe_toxic, obscene, insult). As a result, **the model often responds to the presence of strong profanity by assigning high scores to several related labels simultaneously, rather than distinguishing the specific function of the word (e.g. insult versus obscenity)**.

This overlap is particularly evident for closely related categories such as **toxic versus severe_toxic**, where differences in intensity are captured only by a small number of extreme terms (e.g. bitch, asshole), rather than by broader linguistic structure or tone.

3.**Lack of context limits discrimination between similar semantic categories**

The feature analysis further **reveals limitations in distinguishing categories that depend on context or target**. For instance, slurs such as faggot appear in both insult and identity_hate, reflecting the model’s inability to determine whether an offensive term is directed at an individual or at a protected group. Similarly, the threat classifier occasionally assigns weight to non-threatening tokens that co-occur with violent language, highlighting the absence of compositional understanding.

These limitations stem from the linear, context-independent nature of the model, which operates solely on surface-level lexical patterns and cannot capture syntactic relations, targets of abuse, or pragmatic intent.

Overall, the interpretability analysis shows that TF-IDF + Logistic Regression learns intuitive and transparent word-level indicators of toxicity, which explains its strong performance on explicit and frequent categories. At the same time, the heavy reliance on surface lexical cues limits its ability to reliably separate closely related toxicity subtypes and to handle context-dependent distinctions.


# 4.DistilBERT fine-tuning

Following the statistical baseline, we next fine-tune a transformer-based model to examine whether contextual representations can address the limitations observed for TF-IDF + Logistic Regression. In particular, we adopt DistilBERT for multi-label classification.

We choose DistilBERT as a compact yet expressive representative of transformer models. Compared to BERT-base, DistilBERT retains most of the contextual modelling capacity while using significantly fewer parameters, making it computationally efficient and sufficient for the relatively short comments and moderate dataset size of the Jigsaw corpus. Larger transformer variants or prompt-based large language models are not considered, as this task is a standard supervised multi-label classification problem with abundant labelled data and strict evaluation metrics, where fine-tuned encoder models offer more stable and controlled behaviour.


## 4.1 Data Loading and Formatting

We keep the dataset split and evaluation metrics fixed, and only change the modelling paradigm. The logistic regression baseline uses TF-IDF features, whereas DistilBERT directly learns contextual representations from tokenized text.

In [1]:
!pip -q install transformers datasets accelerate #load model

import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import f1_score, classification_report

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# ----------------------------
# 1) Load train/test (same as LR)
# ----------------------------
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/toxic_comment_dataset"

train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
test_df  = pd.read_csv(f"{DATA_DIR}/test.csv")

label_cols = ["toxic","severe_toxic","obscene","threat","insult","identity_hate"]

train_df["comment_text"] = train_df["comment_text"].fillna("")
test_df["comment_text"]  = test_df["comment_text"].fillna("")

# HuggingFace Dataset needs a single "labels" field for multi-label classification
train_df["labels"] = train_df[label_cols].values.tolist()
test_df["labels"]  = test_df[label_cols].values.tolist()

train_ds = Dataset.from_pandas(train_df[["comment_text","labels"]])
test_ds  = Dataset.from_pandas(test_df[["comment_text","labels"]])

# Cast labels to float type for BCEWithLogitsLoss
train_ds = train_ds.map(lambda e: {'labels': [float(l) for l in e['labels']]})
test_ds = test_ds.map(lambda e: {'labels': [float(l) for l in e['labels']]})

Map:   0%|          | 0/159571 [00:00<?, ? examples/s]

Map:   0%|          | 0/63978 [00:00<?, ? examples/s]

The labels are explicitly cast from integers (0/1) to floating-point values to match the expected input type of the `BCEWithLogitsLoss` used for multi-label classification. Unlike scikit-learn’s Logistic Regression, where label handling is abstracted away, transformer models operate directly on tensor-based loss functions and therefore require explicit type alignment.

## 4.2 Training Setup

#### 4.2.1 Inspecting Comment Lengths to Set the Input Sequence Length

In [4]:
# comment_text length
char_lengths = train_df["comment_text"].str.len()

print(char_lengths.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))

count    159571.000000
mean        394.073221
std         590.720282
min           6.000000
50%         205.000000
75%         435.000000
90%         889.000000
95%        1355.000000
99%        3444.000000
max        5000.000000
Name: comment_text, dtype: float64


In [5]:
from transformers import AutoTokenizer
import numpy as np

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Sample a subset of comments for efficiency
sample_texts = train_df["comment_text"].sample(20000, random_state=42)

token_lengths = [
    len(tokenizer.tokenize(text))
    for text in sample_texts
]

token_lengths = np.array(token_lengths)

print("Token length statistics:")
print(np.percentile(token_lengths, [50, 75, 90, 95, 99]))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (797 > 512). Running this sequence through the model will result in indexing errors


Token length statistics:
[ 50.  99. 198. 298. 775.]


The token length distribution shows that most comments are relatively short: approximately half contain fewer than 50 tokens, and around 75% are shorter than 100 tokens. About **90% of comments fall below 200 tokens**, while very long comments are rare and can be treated as outliers. Based on this distribution, we set the maximum input length to **192 tokens**, which covers the majority of samples while avoiding the unnecessary computational cost associated with larger sequence lengths such as 256. This choice represents a practical trade-off between coverage and efficiency and follows common practice in real-world transformer-based text classification.

#### 4.2.2 Model Training

In [6]:
# ----------------------------
# 2) Tokenization (replaces TF-IDF manual feature)
# ----------------------------
checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize(batch):
    return tokenizer(
        batch["comment_text"],
        truncation=True,
        padding="max_length",
        max_length=192  # comments are short; 128-256 is common
    )

train_ds = train_ds.map(tokenize, batched=True)
test_ds  = test_ds.map(tokenize, batched=True)

train_ds = train_ds.remove_columns(["comment_text"])
test_ds  = test_ds.remove_columns(["comment_text"])

train_ds.set_format("torch")
test_ds.set_format("torch")

Map:   0%|          | 0/159571 [00:00<?, ? examples/s]

Map:   0%|          | 0/63978 [00:00<?, ? examples/s]

In [7]:
# ----------------------------
# 3) Model: DistilBERT with multi-label head
# ----------------------------
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=len(label_cols),
    problem_type="multi_label_classification"
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
# ----------------------------
# 4) Metrics (same idea as LR)
# ----------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))          # sigmoid
    preds = (probs >= 0.5).astype(int)         # threshold 0.5 baseline

    micro = f1_score(labels, preds, average="micro", zero_division=0)
    macro = f1_score(labels, preds, average="macro", zero_division=0)
    return {"micro_f1": micro, "macro_f1": macro}

In [ ]:
# ----------------------------
# 5) Train
# ----------------------------
import torch
from transformers import DataCollatorWithPadding

# Define a custom DataCollator to ensure labels are of type torch.float32
class CustomDataCollator(DataCollatorWithPadding):
    def __call__(self, features):
        batch = super().__call__(features)
        # Ensure labels are float32 for BCEWithLogitsLoss
        if "labels" in batch:
            batch["labels"] = batch["labels"].to(torch.float32)
        return batch

training_args = TrainingArguments(
    output_dir="./distilbert_jigsaw",
    eval_strategy="epoch",
    save_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=200,
    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    data_collator=CustomDataCollator(tokenizer=tokenizer) # Use the custom data collator
)

trainer.train()

/tmp/ipython-input-302459142.py:29: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"


Epoch,Training Loss,Validation Loss,Micro F1,Macro F1
1,0.035200,0.063662,0.682648,0.586517
2,0.032100,0.068226,0.682864,0.617464


TrainOutput(global_step=19948, training_loss=0.03847226894329707, metrics={'train_runtime': 1901.1897, 'train_samples_per_second': 167.864, 'train_steps_per_second': 10.492, 'total_flos': 1.5854597349474816e+16, 'train_loss': 0.03847226894329707, 'epoch': 2.0})

The DistilBERT model is fine-tuned using standard hyperparameters commonly adopted in transformer-based text classification, including a learning rate of 2e-5, a batch size of 16, and two training epochs. Given the large dataset size, the simplicity of the language, and the use of a pretrained transformer, two fine-tuning epochs are sufficient and help reduce the risk of overfitting, especially for rare labels.

# 5.Overall Performance Comparison

In [ ]:
# ----------------------------
# 6) Evaluate + per-label report
# ----------------------------
pred = trainer.predict(test_ds)
logits = pred.predictions
labels = pred.label_ids

probs = 1 / (1 + np.exp(-logits))
preds = (probs >= 0.5).astype(int)

print("Micro-F1:", f1_score(labels, preds, average="micro", zero_division=0))
print("Macro-F1:", f1_score(labels, preds, average="macro", zero_division=0))
print("\nPer-label performance:\n")
print(classification_report(labels, preds, target_names=label_cols, zero_division=0))

Micro-F1: 0.6828639371973756
Macro-F1: 0.6174638098955676

Per-label performance:

               precision    recall  f1-score   support

        toxic       0.54      0.91      0.68      6090
 severe_toxic       0.42      0.41      0.41       367
      obscene       0.62      0.81      0.71      3691
       threat       0.55      0.56      0.55       211
       insult       0.65      0.77      0.71      3427
identity_hate       0.66      0.64      0.65       712

    micro avg       0.58      0.82      0.68     14498
    macro avg       0.57      0.68      0.62     14498
 weighted avg       0.59      0.82      0.68     14498
  samples avg       0.08      0.08      0.08     14498



Overall, fine-tuning DistilBERT leads to a moderate improvement in Micro-F1 (from 0.64 to 0.68) but a substantially larger improvement in Macro-F1 (0.49 to 0.62) compared to the TF-IDF + Logistic Regression baseline.

While the Micro-F1 score increases only slightly, the Macro-F1 score shows a much larger gain, indicating that the main benefit of **DistilBERT** does not come from improving already well-performing frequent labels, but from **better handling rare and difficult labels under severe class imbalance**.

This difference is **primarily driven by a structural change in recall behaviour**. The TF-IDF + Logistic Regression model exhibits a conservative decision pattern, with relatively high precision but low recall, especially for rare labels. In practice, this means that the model often defaults to predicting the negative class when uncertain, leading to many false negatives. For example, the recall for the *threat* and *identity_hate* labels remains very low under the statistical baseline.

In contrast, DistilBERT shows a markedly different behaviour: recall increases substantially across nearly all labels, with particularly strong gains for rare and context-dependent categories.

(1) For *threat*, recall increases from 0.21 to 0.56, nearly doubling performance. This reflects the fact that threats often cannot be identified by single keywords alone but require understanding short contextual patterns such as verb–object relations (e.g. “I will kill you”).

(2) For *identity_hate*, recall rises from 0.25 to 0.64, highlighting the advantage of contextual representations in distinguishing identity-targeted hate from general insults, where recognising who is being targeted is crucial.

(3) For *severe_toxic*, improvements are smaller and more limited, which is consistent with the ambiguous definition of this label and its low frequency; here, data noise and label subjectivity appear to be stronger bottlenecks than model capacity.

(4) For frequent labels such as *toxic*, *obscene*, and *insult*, DistilBERT still improves performance, but the gains are more modest, as these categories already rely heavily on explicit lexical cues that the TF-IDF baseline captures well.

Importantly, these improvements are not simply the result of threshold tuning, but reflect a genuine change in modelling capacity. Whereas the statistical baseline relies primarily on surface-level keyword presence, DistilBERT combines lexical information with local contextual structure, leading to higher recall on rare and subtle forms of toxic language at the cost of a moderate reduction in precision. This trade-off is particularly relevant in content moderation settings, where missing harmful content is often more problematic than producing false positives.

## 5.1.Error Analysis & Interpretability Comparison

While linear models allow direct inspection of feature weights, this approach is no longer applicable to transformer-based architectures. Instead, we rely on **post-hoc attribution methods** such as Integrated Gradients, combined with error-type analysis and qualitative case studies. Together, these methods provide a comparable level of interpretability, albeit at the cost of transparency, reflecting a general trade-off between model capacity and interpretability.

To better understand how DistilBERT differs from the TF-IDF baseline, we conduct a combined error analysis and interpretability study. In particular, we focus on three questions.

1. When DistilBERT correctly identifies rare labels such as *threat* and *identity_hate*, what textual cues does it rely on, and does it indeed attend to meaningful phrases and local context rather than isolated keywords?  

2. Is the observed performance gain mainly driven by a reduction in false negatives, reflecting improved recall on difficult cases?

3. Does this improvement come at the cost of increased false positives, revealing a trade-off between recall and precision that is characteristic of neural black-box models?

## 5.2. Behavioural Difference: TP / FP / FN per label

In [ ]:
# ----------------------------
# 7. Interpretability
# Error Decomposition: TP / FP / FN per label
# ----------------------------

# Ensure numpy arrays
y_true = Y_test.values.astype(int)
preds_lr = Y_pred.astype(int)
preds_db = preds.astype(int)

def per_label_counts(y_true, y_pred, label_names):
    """
    Compute TP / FP / FN for each label.
    """
    stats = {}
    for i, label in enumerate(label_names):
        yt = y_true[:, i]
        yp = y_pred[:, i]

        stats[label] = {
            "TP": int(((yt == 1) & (yp == 1)).sum()),
            "FP": int(((yt == 0) & (yp == 1)).sum()),
            "FN": int(((yt == 1) & (yp == 0)).sum())
        }
    return stats

# Compute counts
counts_lr = per_label_counts(y_true, preds_lr, label_cols)
counts_db = per_label_counts(y_true, preds_db, label_cols)

# Display comparison
for label in label_cols:
    print(
        f"{label:15s} "
        f"LR: {counts_lr[label]} "
        f"DistilBERT: {counts_db[label]}"
    )

toxic LR: {'TP': 4266, 'FP': 2290, 'FN': 1824} DistilBERT: {'TP': 5549, 'FP': 4726, 'FN': 541}
severe_toxic LR: {'TP': 115, 'FP': 165, 'FN': 252} DistilBERT: {'TP': 149, 'FP': 210, 'FN': 218}
obscene LR: {'TP': 2276, 'FP': 740, 'FN': 1415} DistilBERT: {'TP': 3006, 'FP': 1806, 'FN': 685}
threat LR: {'TP': 44, 'FP': 54, 'FN': 167} DistilBERT: {'TP': 118, 'FP': 97, 'FN': 93}
insult LR: {'TP': 1755, 'FP': 652, 'FN': 1672} DistilBERT: {'TP': 2642, 'FP': 1419, 'FN': 785}
identity_hate LR: {'TP': 178, 'FP': 72, 'FN': 534} DistilBERT: {'TP': 453, 'FP': 230, 'FN': 259}


Across almost all labels, the main performance gain of DistilBERT over Logistic Regression (TF-IDF) does not come from reducing false positives (FP), but from **substantially reducing false negatives (FN)**. In the context of content moderation, this shift is critical, since FN (missed harmful content) is generally more dangerous than FP (over-blocking benign content).

---

Overall Toxicity & Common Abuse (toxic, obscene, insult)

For high-frequency but context-dependent labels such as *toxic,obscene, and insult*, DistilBERT exhibits a consistent pattern:

1. FN drops sharply, while TP increases significantly.
2. FP also increases, indicating a more aggressive decision boundary.
This suggests that DistilBERT is less hesitant to assign the positive class when contextual cues indicate harmful intent. Unlike Logistic Regression, which relies heavily on explicit keywords, DistilBERT captures implicit abuse, contextual vulgarity, and indirect insults, leading to much higher recall. The cost of this behavior is lower precision, as some borderline or ambiguous cases are flagged as toxic.

---
Rare and High-Risk Labels (threat, identity_hate)

The most decisive improvements appear in rare but high-risk categories, especially threat and identity_hate:

1. FN is reduced by nearly half.
2. TP increases dramatically, often by a much larger margin than FP.

These labels cannot be reliably detected by surface lexical features alone. DistilBERT benefits from contextual understanding, such as recognizing: Action–target structures (e.g., threats involving intent and victims),whether an insult is directed at an individual vs. a protected group. This behavior highlights the core advantage of contextual models: they do not merely detect offensive words, but who is being targeted and how, which is essential for fine-grained moderation.

---
Ambiguous Severity Labels (severe_toxic)

In contrast, *severe_toxic* shows only marginal improvement: FN decreases slightly, FP increases slightly, but overall gains are limited. This suggests that model capacity is not the primary bottleneck here. Instead, the label itself has unclear semantic boundaries and higher annotation noise, making it difficult even for contextual models to learn a stable decision rule.

Logistic Regression (TF-IDF) is conservative: it produces fewer FP but many FN. It performs adequately for explicit profanity but fails on rare, implicit, or context-dependent toxicity.

DistilBERT adopts a more aggressive stance: it significantly reduces FN—especially for rare and semantically complex labels—at the cost of higher FP. This trade-off leads to higher recall but lower precision and reduced interpretability.

Overall, the error decomposition shows that DistilBERT’s improvement is structural rather than incremental: it shifts the error profile toward risk-aware moderation, prioritizing the detection of harmful content even when linguistic signals are subtle or indirect.

## 5.3 Interpretability: Integrated Gradients

Integrated Gradients is used to provide token-level attribution for a given label prediction, allowing us to **inspect which parts of a comment contribute most to the model’s decision**. Rather than serving as a generic explainability tool, IG is applied here to support qualitative analysis of representative cases, helping us understand how the model interprets harmful language and which textual cues drive correct predictions, false positives, or false negatives.

In [ ]:
# ----------------------------
# 7. Interpretability: Integrated Gradients
# ----------------------------

import torch
import numpy as np
from captum.attr import LayerIntegratedGradients

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = trainer.model
model.eval().to(device)

label_to_idx = {l: i for i, l in enumerate(label_cols)}

def explain_with_ig(text, label_name, max_length=192, top_k=15):
    """
    Token-level attribution using Integrated Gradients.
    """
    target_idx = label_to_idx[label_name]

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    ).to(device)

    with torch.no_grad():
        logits = model(**inputs).logits.squeeze(0)
        prob = torch.sigmoid(logits[target_idx]).item()

    def forward_func(input_ids, attention_mask):
        return model(
            input_ids=input_ids.long(),
            attention_mask=attention_mask
        ).logits[:, target_idx]

    lig = LayerIntegratedGradients(
        forward_func,
        model.distilbert.embeddings.word_embeddings
    )

    baseline = torch.full_like(inputs["input_ids"], tokenizer.pad_token_id)

    attributions = lig.attribute(
        inputs=inputs["input_ids"],
        baselines=baseline,
        additional_forward_args=(inputs["attention_mask"],)
    )

    scores = attributions.sum(dim=-1).squeeze(0).cpu().numpy()
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"].squeeze(0))

    token_scores = sorted(
        [(t, s) for t, s in zip(tokens, scores) if t not in ["[CLS]", "[SEP]", "[PAD]"]],
        key=lambda x: abs(x[1]),
        reverse=True
    )

    print(f"\nLabel: {label_name} | Probability: {prob:.3f}")
    for tok, score in token_scores[:top_k]:
        print(f"{tok:>12s}  {score:+.4f}")

### 5.3.1.Selecting Representative Cases

Instead of analyzing all samples, we deliberately select representative instances where the model makes confident or critical decisions, such as strong true positives, false positives, and near-miss false negatives. The goal is not statistical coverage, but to better understand how the model reasons about the input text when producing these predictions.

In [ ]:
# --------------------------------------------------
# Assumed available variables from previous sections
# --------------------------------------------------
# test_df   : DataFrame containing "comment_text" and label columns
# label_cols: list of label names
# preds_lr  : ndarray (N, 6), binary predictions from TF-IDF + LR
# preds     : ndarray (N, 6), binary predictions from DistilBERT
# probs     : ndarray (N, 6), predicted probabilities from DistilBERT

preds_db = preds
probs_db = probs

# Ground-truth labels
y_true = test_df[label_cols].values.astype(int)

label_to_idx = {label: i for i, label in enumerate(label_cols)}

# --------------------------------------------------
# Error-type based sample selection
# --------------------------------------------------
def pick_indices(case_name, label_name, k=3, seed=42):
    """
    Select representative sample indices for a given error type and label.

    Cases:
        - FN_LR_to_TP_DB: False Negative for LR, corrected by DistilBERT
        - FP_DB         : False Positive for DistilBERT
        - FN_DB         : False Negative for DistilBERT
        - TP_DB         : True Positive for DistilBERT (optional reference)
    """
    i = label_to_idx[label_name]
    yt  = y_true[:, i]
    lr  = preds_lr[:, i]
    db  = preds_db[:, i]

    if case_name == "FN_LR_to_TP_DB":
        idx = np.where((yt == 1) & (lr == 0) & (db == 1))[0]
    elif case_name == "FP_DB":
        idx = np.where((yt == 0) & (db == 1))[0]
    elif case_name == "FN_DB":
        idx = np.where((yt == 1) & (db == 0))[0]
    elif case_name == "TP_DB":
        idx = np.where((yt == 1) & (db == 1))[0]
    else:
        raise ValueError("Unknown case_name")

    if len(idx) == 0:
        print(f"[WARN] No samples found for {case_name} / {label_name}")
        return []

    # Prefer samples with higher DistilBERT confidence
    if probs_db is not None:
        p = probs_db[idx, i]
        order = np.argsort(-p)
        idx = idx[order[:k]]
        return idx.tolist()

    # Fallback: random selection
    rng = np.random.default_rng(seed)
    if len(idx) > k:
        idx = rng.choice(idx, size=k, replace=False)
    return idx.tolist()


# --------------------------------------------------
# Display selected samples
# --------------------------------------------------
def show_samples(indices, label_name, max_chars=280):
    """
    Display a short preview of selected samples for qualitative inspection.
    """
    i = label_to_idx[label_name]
    rows = []

    for j in indices:
        text = test_df.iloc[j]["comment_text"]
        rows.append({
            "index": j,
            "true_label": int(y_true[j, i]),
            "LR_pred": int(preds_lr[j, i]),
            "DistilBERT_pred": int(preds_db[j, i]),
            "DistilBERT_prob": float(probs_db[j, i]) if probs_db is not None else None,
            "text_preview": (
                text[:max_chars] + "…" if len(text) > max_chars else text
            )
        })

    return pd.DataFrame(rows)

# --------------------------------------------------
# Key cases for qualitative analysis
# --------------------------------------------------
picked = {}

# 1) Largest improvements: FN (LR) → TP (DistilBERT)
picked[("FN_LR_to_TP_DB", "threat")] = pick_indices(
    "FN_LR_to_TP_DB", "threat", k=3
)
picked[("FN_LR_to_TP_DB", "identity_hate")] = pick_indices(
    "FN_LR_to_TP_DB", "identity_hate", k=3
)

# 2) Cost of higher recall: false positives by DistilBERT
picked[("FP_DB", "toxic")] = pick_indices(
    "FP_DB", "toxic", k=3
)

# 3) Remaining failure cases: false negatives by DistilBERT
picked[("FN_DB", "severe_toxic")] = pick_indices(
    "FN_DB", "severe_toxic", k=3
)

# --------------------------------------------------
# Preview selected samples
# --------------------------------------------------
for (case_name, label_name), idxs in picked.items():
    print("\n" + "=" * 90)
    print(f"CASE: {case_name} | LABEL: {label_name} | n={len(idxs)}")
    if len(idxs) > 0:
        display(show_samples(idxs, label_name))


CASE: FN_LR_to_TP_DB | LABEL: threat | n=3


,index,true,LR_pred,DB_pred,DB_prob,text_preview
0,7121,1,0,1,0.867598,"I will hunt you down and kill you, Nazi filth."
1,61110,1,0,1,0.861655,I KILLYOU!!!!! YOU DELETE MY PAGE??? THAT NI B...
2,39681,1,0,1,0.858363,I am going to rip off your tiny balls and deca...



CASE: FN_LR_to_TP_DB | LABEL: identity_hate | n=3


,index,true,LR_pred,DB_pred,DB_prob,text_preview
0,38788,1,0,1,0.926170,"YOU GAY PRICK, WHY THE FUCK DID U REVERT MY ED..."
1,5192,1,0,1,0.920218,DIS NIGGA IS GAY HE KILLED LOTS OF FUQIN PPL T...
2,19074,1,0,1,0.919931,"GAY FAGS SPREAD AIDS AND FUCK LITTLE KIDS, STO..."



CASE: FP_DB | LABEL: toxic | n=3


,index,true,LR_pred,DB_pred,DB_prob,text_preview
0,8080,0,1,1,0.998750,BOMB IN THE ASSHOLE
1,49197,0,1,1,0.998659,Does it even fucking matter??? What you just w...
2,34261,0,1,1,0.998606,== \n == Headline text == \n hahaha bitches!!...



CASE: FN_DB | LABEL: severe_toxic | n=3


,index,true,LR_pred,DB_pred,DB_prob,text_preview
0,34078,1,0,0,0.499724,Go fuck yourself slav.
1,18490,1,1,0,0.498429,"== hiya... == \n\n hey aussie fag, how would y..."
2,6723,1,0,0,0.497910,Nigger alert! I fucked a nigger last night! ;D...


We analyze representative cases to characterize DistilBERT’s behavior across different toxicity labels. Specifically, we examine instances where DistilBERT corrects false negatives made by logistic regression (threat, identity_hate), as well as cases where false negatives remain (severe_toxic). These examples illustrate both the strengths and limitations of contextual modeling beyond aggregate metrics.

1. Threat (DistilBERT improvement)
“I will hunt you down and kill you, Nazi filth.”
This case demonstrates DistilBERT’s ability to capture explicit violent intent directed at a specific target, allowing it to correctly identify threats that are missed by linear models focusing on isolated keywords.

2. Identity Hate (DistilBERT improvement)
“GAY FAGS SPREAD AIDS AND FUCK LITTLE KIDS, STO…”
Here, DistilBERT correctly identifies identity-targeted hate by focusing on repeated group references rather than generic profanity, highlighting its strength in modeling who is being attacked, not just how offensive the language is.

3. Severe Toxic (DistilBERT failure)
“Go fuck yourself slav.”
Despite recognizing strong profanity, the model produces a near-boundary prediction (≈0.49), reflecting uncertainty about severity. This illustrates a key challenge of the severe_toxic label, which often depends on implicit intensity and escalation rather than explicit lexical cues, rather than a simple failure of contextual understanding.

In [ ]:
j = 7121
explain_with_ig(test_df.iloc[j]["comment_text"], "threat")

TEXT:
I will hunt you down and kill you, Nazi filth.
TARGET LABEL: threat
Logit: 1.8799
Probability: 0.8676

Top contributing tokens:
        kill  +2.0986
        nazi  +1.2802
         you  +1.2341
         and  +1.2300
         you  +0.8447
        will  +0.6791
           i  +0.6121
        down  +0.4470
           .  +0.3090
           ,  +0.2963
        hunt  +0.2604
          fi  +0.2075
       ##lth  +0.1766


The attribution results show that the prediction is primarily driven by explicit violent action verbs, with “kill” receiving the strongest contribution. The repeated high attribution of “you” indicates that the model recognizes a directed threat toward a specific target rather than a generic insult. Importantly, the model leverages the full action sequence (“will” → “hunt” → “down” → “kill”), suggesting contextual understanding of intent and future violence. While “nazi” contributes to the prediction, it is not the dominant signal, indicating that the model does not rely solely on identity-related terms. This explains why DistilBERT correctly identifies the instance as a threat, whereas the linear baseline fails to capture the explicit violent intent.

In [ ]:
j = 19074
explain_with_ig(test_df.iloc[j]["comment_text"], "identity_hate")

TEXT:
GAY FAGS SPREAD AIDS AND FUCK LITTLE KIDS, STONE AND GAS THE FAGS. GAY FAGS FUCK THE WORLD AND GOD HATES FAGS. GAY FAGS SPREAD AIDS AND FUCK LITTLE KIDS, STONE AND GAS THE FAGS. GAY FAGS FUCK THE WORLD AND GOD HATES FAGS. GAY FAGS SPREAD AIDS AND FUCK LITTLE KIDS, STONE AND GAS THE FAGS. GAY FAGS FUCK THE WORLD AND GOD HATES FAGS. GAY FAGS SPREAD AIDS AND FUCK LITTLE KIDS, STONE AND GAS THE FAGS. GAY FAGS FUCK THE WORLD AND GOD HATES FAGS. GAY FAGS SPREAD AIDS AND FUCK LITTLE KIDS, STONE AND GAS THE FAGS. GAY FAGS FUCK THE WORLD AND GOD HATES FAGS. GAY FAGS SPREAD AIDS AND FUCK LITTLE KIDS, STONE AND GAS THE FAGS. GAY FAGS FUCK THE WORLD AND GOD HATES FAGS. GAY FAGS SPREAD AIDS AND FUCK LITTLE KIDS, STONE AND GAS THE FAGS. GAY FAGS FUCK THE WORLD AND GOD HATES FAGS. GAY FAGS SPREAD AIDS AND FUCK LITTLE KIDS, STONE AND GAS THE FAGS. GAY FAGS FUCK THE WORLD AND GOD HATES FAGS. GAY FAGS SPREAD AIDS AND FUCK LITTLE KIDS, STONE AND GAS THE FAGS. GAY FAGS FUCK THE WORLD AND GOD HATES F

The attribution results are dominated by repeated occurrences of the identity term “gay”, indicating that the model treats group identity references as the primary evidence for the identity_hate label. Although the slur “fags” is split into subword tokens, these components still receive positive attribution, showing that WordPiece tokenization does not prevent the model from capturing the underlying hateful expression. Notably, generic profanity does not dominate the attribution pattern, suggesting that the model distinguishes identity-targeted hate from general insults. The high prediction confidence further indicates that DistilBERT strongly associates repeated group-directed attacks with identity-based hate, explaining its improvement over the linear baseline.

In [ ]:
j = 34078
explain_with_ig(test_df.iloc[j]["comment_text"], "severe_toxic")

TEXT:
Go fuck yourself slav.
TARGET LABEL: severe_toxic
Logit: -0.0011
Probability: 0.4997

Top contributing tokens:
        fuck  +6.1195
          sl  +1.5744
          go  +1.4350
    yourself  +1.2492
        ##av  +0.2983
           .  +0.2891


The attribution is dominated by the profanity “fuck”, indicating that the model clearly recognizes strong offensive language. However, despite this high contribution, the overall logit remains close to zero, placing the prediction directly on the decision boundary. This suggests that while the model detects explicit insults, it lacks additional escalation signals—such as threats, sustained abuse, or violent intent—that typically distinguish severe_toxic from general toxicity. Importantly, this reflects a limitation of the label definition rather than a failure to process the text, as severity often depends on implicit intensity rather than explicit lexical cues. This helps explain why false negatives for severe_toxic persist even with contextual models.

Integrated Gradients shows that DistilBERT distinguishes toxicity labels using different token-level evidence patterns: threat predictions are driven by explicit future-oriented action verbs combined with a clear target (e.g., “will → hunt → kill” + “you”); identity_hate predictions are dominated by repeated identity references rather than generic profanity; and severe_toxic failures occur despite strong profanity attribution when no escalation or violence cues are present. This indicates that DistilBERT’s improvements are label-specific and mechanism-driven, while remaining errors are tied to ambiguity in severity rather than missing lexical signals.

# 6.Discussion & Conclusion

The comparison between TF-IDF + Logistic Regression and DistilBERT shows a consistent overall performance advantage for the contextual model, with higher recall and F1 scores across most toxicity labels, particularly for categories involving explicit intent or group targeting. However, these gains are uneven across labels, with substantial improvements for *threat* and *identity_hate*, but only marginal gains for *severe_toxic*.

Interpretability analyses across models consistently explain this pattern. Inspection of learned coefficients in the linear model shows that TF-IDF + Logistic Regression relies heavily on isolated lexical indicators, such as profanity or slur-related terms, which leads to reasonable performance on generic toxic content but frequent false negatives when intent, targets, or relational structure are required. This limitation is further reflected in error analysis, where the linear model systematically misses threats expressed through action sequences or identity-directed attacks lacking distinctive keywords.

In contrast, DistilBERT’s behavior, as revealed by Integrated Gradients and case-based error analysis, demonstrates reliance on structured semantic cues. For *threat*, predictions are driven by combinations of future-oriented violent actions and explicit targets; for *identity_hate*, repeated group identity references dominate over generic profanity. These mechanisms directly correspond to the observed recall improvements over the linear baseline.

However, interpretability analysis also reveals shared limitations across models for *severe_toxic*. Both coefficient-based inspection and token-level attributions show that while strong profanity is consistently recognized, predictions remain uncertain in the absence of explicit escalation, threats, or sustained abuse. This indicates that residual errors are primarily due to ambiguity in the label’s severity definition rather than insufficient feature representation or contextual modeling.

Overall, the combined performance results, model-specific interpretability analyses, and systematic error inspection indicate that contextual models outperform linear baselines when classification depends on relational semantics—such as intent, targets, or group-directedness—but offer limited advantage when labels hinge on implicit intensity judgments that are not explicitly encoded in the text.